# Single-Stock Analysis Playbook Template (OpenBB + FinanceToolkit)

This notebook implements the full strategy in `Analysis/docs/SINGLE_STOCK_ANALYSIS_STRATEGY.md` with all phases in one place.

**Primary data source:** OpenBB `provider="fmp_cached"`  
**Fallback:** OpenBB `provider="fmp"` if a specific endpoint fails under cached mode.

## Phases
1. Company Profile & Quality
2. 5-Year Fundamentals
3. Technical Analysis & Timing
4. Valuation & Fair Value
5. Risk & Portfolio Context
6. Market Segment ETF + Peer Relative Analysis
7. Decision, Execution, and Monitoring

## Notebook Setup
Load OpenBB and FinanceToolkit from editable source installs and initialize environment credentials.

### Initial setup checklist (run once per environment)
1. Open this repository as the workspace root: `I:\masterswork\git\OpenBB`.
2. Use the project Python environment (`.venv_win`) as the notebook kernel.
3. Install editable source packages for local development:
   - `openbb_platform/core`
   - `openbb_platform/extensions/equity`
   - `openbb_platform/extensions/etf`
   - `openbb_platform/providers/fmp`
   - `openbb_platform/providers/fmp_cached`
   - `openbb_platform/providers/yfinance`
   - `FinanceToolkit`
4. Ensure `.env` exists at `I:\masterswork\git\OpenBB\.env` with `FMP_API_KEY` (or `FMP_API`).
5. Verify `seaborn` is installed in the same environment for Phase 6 visuals.

### Run order for each new session
1. Run Cell 3 (imports + credentials setup).
2. Run Cell 5 (configuration inputs).
3. Run Cell 7 (helper functions).
4. Run remaining cells in order from Phase 1 to Phase 7.

### Maintenance rules
- Keep `PRIMARY_PROVIDER = "fmp_cached"` and `FALLBACK_PROVIDER = "fmp"`.
- If provider or schema behavior changes, update helper logic in Cell 7 first.
- Re-run all cells after any package update or kernel restart.

In [14]:
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# Editable installs are now expected for OpenBB + FinanceToolkit.
from openbb import obb
import openbb
from financetoolkit import Toolkit

# Environment variables
load_dotenv(r"I:\masterswork\git\OpenBB\.env", override=True)
fmp_api_key = os.getenv("FMP_API_KEY") or os.getenv("FMP_API")

if fmp_api_key:
    obb.user.credentials.fmp_api_key = fmp_api_key
    obb.user.credentials.fmp_cached_api_key = fmp_api_key

print("Setup complete")
print(f"Python time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"OpenBB loaded from: {openbb.__file__}")
print(f"FinanceToolkit loaded from: {Toolkit.__module__}")
print(f"FMP key loaded: {'YES' if bool(fmp_api_key) else 'NO'}")
print("Primary provider = fmp_cached")

Setup complete
Python time: 2026-02-28 00:41:02
OpenBB loaded from: I:\masterswork\git\OpenBB\openbb_platform\core\openbb\__init__.py
FinanceToolkit loaded from: financetoolkit.toolkit_controller
FMP key loaded: YES
Primary provider = fmp_cached


## Configuration Inputs
Set ticker, benchmark, date ranges, and provider preferences.

In [ ]:
SYMBOL = "CLS"
BENCHMARK = "SPY"

today = pd.Timestamp.today().normalize()
START_DATE_FUNDAMENTALS = (today - pd.DateOffset(years=5)).strftime("%Y-%m-%d")
START_DATE_TECHNICALS = (today - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
END_DATE = today.strftime("%Y-%m-%d")

PRIMARY_PROVIDER = "fmp_cached"
FALLBACK_PROVIDER = "fmp"
RISK_FREE_RATE = 0.02

load_dotenv(r"I:\\masterswork\\git\\OpenBB\\.env")
API_KEY = os.getenv("FMP_API_KEY")

obb.user.preferences.output_type = "dataframe"

print({
    "symbol": SYMBOL,
    "benchmark": BENCHMARK,
    "fundamental_start": START_DATE_FUNDAMENTALS,
    "technical_start": START_DATE_TECHNICALS,
    "end_date": END_DATE,
    "primary_provider": PRIMARY_PROVIDER,
    "fallback_provider": FALLBACK_PROVIDER,
    "api_key_loaded": API_KEY is not None
})

{'symbol': 'CLS', 'benchmark': 'SPY', 'fundamental_start': '2021-01-01', 'technical_start': '2024-01-01', 'end_date': '2026-02-28', 'primary_provider': 'fmp_cached', 'fallback_provider': 'fmp', 'api_key_loaded': True}


## Helper Functions
Utility helpers for provider fallback, dataframe normalization, and scalar extraction.

In [15]:
def to_df(obj):
    if obj is None:
        return pd.DataFrame()
    if hasattr(obj, "to_df"):
        df = obj.to_df()
        if isinstance(df, pd.DataFrame):
            return df
    if isinstance(obj, pd.DataFrame):
        return obj
    return pd.DataFrame()


def call_obb(endpoint, *args, provider_priority=None, **kwargs):
    providers = provider_priority or [PRIMARY_PROVIDER, FALLBACK_PROVIDER]
    errors = []
    for provider in providers:
        try:
            return endpoint(*args, provider=provider, **kwargs), provider
        except Exception as exc:
            errors.append((provider, str(exc)))
    error_text = "; ".join([f"{p}: {e}" for p, e in errors])
    raise RuntimeError(f"All providers failed for {endpoint}: {error_text}")


def first_available_value(df, candidates, default=None):
    if df is None or df.empty:
        return default
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        key = c.lower()
        if key in lower_map:
            series = df[lower_map[key]].dropna()
            if len(series):
                return series.iloc[0]
    return default


def last_numeric(series_or_df):
    if isinstance(series_or_df, pd.DataFrame):
        numeric = series_or_df.select_dtypes(include=["number"])
        if numeric.empty:
            return np.nan
        vals = numeric.tail(1).values.flatten()
        vals = [v for v in vals if pd.notna(v)]
        return vals[0] if vals else np.nan
    if isinstance(series_or_df, pd.Series):
        ser = pd.to_numeric(series_or_df, errors="coerce").dropna()
        return ser.iloc[-1] if len(ser) else np.nan
    return np.nan

## Phase 1 — Company Profile & Business Quality

In [16]:
profile_obj, profile_provider = call_obb(obb.equity.profile, symbol=SYMBOL)
quote_obj, quote_provider = call_obb(obb.equity.price.quote, symbol=SYMBOL)
metrics_obj, metrics_provider = call_obb(obb.equity.fundamental.metrics, symbol=SYMBOL, period="annual", limit=5)

profile_df = to_df(profile_obj)
quote_df = to_df(quote_obj)
metrics_df = to_df(metrics_obj)

peers_obj, peers_provider = call_obb(obb.equity.compare.peers, symbol=SYMBOL)
peers_df = to_df(peers_obj)

print("Providers used:", {
    "profile": profile_provider,
    "quote": quote_provider,
    "metrics": metrics_provider,
    "peers": peers_provider,
})

display(profile_df.head(1))
display(quote_df.head(1))
display(metrics_df.head(5))
display(peers_df.head(10))

⚠️  FALLBACK: Using fallback implementation for endpoint: EquityInfo
   Consider implementing dedicated database persistence in the specific model file
⚠️  FALLBACK: Using fallback implementation for endpoint: EquityQuote
   Consider implementing dedicated database persistence in the specific model file
⚠️  FALLBACK: Using fallback implementation for endpoint: KeyMetrics
   Consider implementing dedicated database persistence in the specific model file
⚠️  FALLBACK: Using fallback implementation for endpoint: EquityPeers
   Consider implementing dedicated database persistence in the specific model file
Providers used: {'profile': 'fmp', 'quote': 'fmp_cached', 'metrics': 'fmp', 'peers': 'fmp'}


,symbol,name,cik,cusip,isin,stock_exchange,long_description,ceo,company_url,business_phone_no,hq_address1,hq_address_city,hq_address_postal_code,hq_state,hq_country,employees,sector,industry_category,first_stock_price_date,is_etf,is_actively_trading,is_adr,is_fund,image,currency,market_cap,last_price,year_high,year_low,volume_avg,annualized_dividend_amount,beta
0,CLS,Celestica Inc.,0001030894,15101Q207,CA15101Q2071,NYSE,Celestica Inc. provides hardware platform and ...,Robert Andrew Mionis,https://www.celestica.com,14164485800,5140 Yonge Street,Toronto,M2N 6L7,ON,CA,21914,Technology,"Hardware, Equipment & Parts",1998-06-30,False,True,False,False,https://images.financialmodelingprep.com/symbo...,USD,31937722310,277.63,363.4,58.05,2733596,0.0,1.505


,symbol,name,exchange,last_price,last_timestamp,open,high,low,volume,prev_close,change,change_percent,year_high,year_low,ma50,ma200,market_cap
0,CLS,Celestica Inc.,NYSE,277.63,2026-02-27 21:00:03,276.99,277.99,270.4642,2179150,279.16,-1.53,-0.005481,363.4,58.05,298.8724,240.12335,3.193772e+10


,symbol,period_ending,fiscal_year,fiscal_period,currency,market_cap,enterprise_value,ev_to_sales,ev_to_operating_cash_flow,ev_to_free_cash_flow,ev_to_ebitda,net_debt_to_ebitda,current_ratio,income_quality,graham_number,graham_net_net,tax_burden,interest_burden,working_capital,invested_capital,return_on_assets,operating_return_on_assets,return_on_tangible_assets,return_on_equity,return_on_invested_capital,return_on_capital_employed,earnings_yield,free_cash_flow_yield,capex_to_operating_cash_flow,capex_to_depreciation,capex_to_revenue,sales_general_and_administrative_to_revenue,research_and_developement_to_revenue,stock_based_compensation_to_revenue,intangibles_to_total_assets,average_receivables,average_payables,average_inventory,days_of_sales_outstanding,days_of_payables_outstanding,days_of_inventory_outstanding,operating_cycle,cash_conversion_cycle,free_cash_flow_to_equity,free_cash_flow_to_firm,tangible_asset_value,net_current_asset_value
0,CLS,2026-02-28,2026,TTM,USD,31937722310,32118293099,2.587749,48.521455,69.892708,26.476217,0.148851,1.440045,0.797969,56.014724,-11.532236,0.846867,0.949229,1730445108,3037462195,0.115784,0.136115,0.126276,0.44134,0.241615,0.28757,0.026111,0.014389,0.305772,1.154917,0.016307,0.0,0.009561,0.004367,0.083085,2.535111e+09,1.770433e+09,2115359084,77.439542,61.965551,72.654534,150.094076,88.128525,278966325.0,7.653613e+08,1614057303,675169053


,symbol,name,price,market_cap
0,SNDK,Sandisk Corporation,635.36,93758327325
1,KEYS,"Keysight Technologies, Inc.",307.33,52804556719
2,GRMN,Garmin Ltd.,252.83,48628009001
3,UI,Ubiquiti Inc.,766.99,46418843790
4,NOK,Nokia Oyj,7.72,41694886240
5,FICO,Fair Isaac Corporation,1409.36,33433026774
6,TDY,Teledyne Technologies Incorporated,681.10,31978318608
7,CTSH,Cognizant Technology Solutions Corporation,64.43,31096911869
8,JBL,Jabil Inc.,264.99,28307015905
9,FLEX,Flex Ltd.,63.02,23304187857


In [ ]:
sector = first_available_value(profile_df, ["sector", "sector_name"], default="Unknown")
industry = first_available_value(profile_df, ["industry", "industry_name"], default="Unknown")
market_cap = first_available_value(quote_df, ["market_cap", "marketcap"], default=np.nan)

phase1_summary = pd.DataFrame([
    {
        "symbol": SYMBOL,
        "sector": sector,
        "industry": industry,
        "market_cap": market_cap,
        "peer_count": len(peers_df)
    }
])
display(phase1_summary)

## Phase 2 — 5-Year Fundamental Analysis

In [ ]:
income_obj, income_provider = call_obb(obb.equity.fundamental.income, symbol=SYMBOL, period="annual", limit=5)
balance_obj, balance_provider = call_obb(obb.equity.fundamental.balance, symbol=SYMBOL, period="annual", limit=5)
cash_obj, cash_provider = call_obb(obb.equity.fundamental.cash, symbol=SYMBOL, period="annual", limit=5)
ratios_obj, ratios_provider = call_obb(obb.equity.fundamental.ratios, symbol=SYMBOL, period="annual", limit=5)

income_df = to_df(income_obj)
balance_df = to_df(balance_obj)
cash_df = to_df(cash_obj)
ratios_df = to_df(ratios_obj)

print("Providers used:", {
    "income": income_provider,
    "balance": balance_provider,
    "cash": cash_provider,
    "ratios": ratios_provider,
})

display(income_df.head())
display(balance_df.head())
display(cash_df.head())
display(ratios_df.head())

In [ ]:
# FinanceToolkit deep ratio extraction
ft = Toolkit(
    [SYMBOL],
    api_key=API_KEY or "",
    start_date=START_DATE_FUNDAMENTALS,
    end_date=END_DATE,
    progress_bar=False,
)
ratios = ft.ratios

fundamental_kpis = {
    "Revenue Growth (last)": last_numeric(ratios.get_revenue_growth()),
    "EPS Growth (last)": last_numeric(ratios.get_earnings_per_share_growth()),
    "FCF Growth (last)": last_numeric(ratios.get_free_cash_flow_growth()),
    "Gross Margin (last)": last_numeric(ratios.get_gross_margin()),
    "Operating Margin (last)": last_numeric(ratios.get_operating_margin()),
    "Net Margin (last)": last_numeric(ratios.get_net_profit_margin()),
    "ROIC (last)": last_numeric(ratios.get_return_on_invested_capital()),
    "Current Ratio (last)": last_numeric(ratios.get_current_ratio()),
    "Debt/Equity (last)": last_numeric(ratios.get_debt_to_equity_ratio()),
}

fundamental_kpi_df = pd.DataFrame.from_dict(fundamental_kpis, orient="index", columns=["value"])
display(fundamental_kpi_df)

## Phase 3 — Technical Analysis & Timing

In [ ]:
ft_ta = Toolkit(
    [SYMBOL],
    api_key=API_KEY or "",
    start_date=START_DATE_TECHNICALS,
    end_date=END_DATE,
    progress_bar=False,
)
tech = ft_ta.technicals

rsi = tech.get_relative_strength_index()
macd = tech.get_moving_average_convergence_divergence()
adx = tech.get_average_directional_index()
bb = tech.get_bollinger_bands()
atr = tech.get_average_true_range()
obv = tech.get_on_balance_volume()

technical_kpis = pd.DataFrame({
    "RSI(14)": [last_numeric(rsi)],
    "ADX(14)": [last_numeric(adx)],
    "ATR": [last_numeric(atr)],
    "OBV": [last_numeric(obv)],
})
display(technical_kpis)

if isinstance(bb, pd.DataFrame) and not bb.empty:
    bb.plot(figsize=(14, 4), title=f"{SYMBOL} Bollinger Bands", grid=True)
    plt.show()

## Phase 4 — Valuation & Fair Value

In [ ]:
models = ft.models

valuation = {
    "P/E (last)": last_numeric(ratios.get_price_earnings_ratio()),
    "EV/EBITDA (last)": last_numeric(ratios.get_ev_to_ebitda()),
    "P/FCF (last)": last_numeric(ratios.get_price_to_free_cash_flow_ratio()),
    "P/S (last)": last_numeric(ratios.get_price_to_sales_ratio()),
    "Earnings Yield (last)": last_numeric(ratios.get_earnings_yield()),
    "WACC (last)": last_numeric(models.get_weighted_average_cost_of_capital()),
    "Piotroski (last)": last_numeric(models.get_piotroski_score()),
    "Altman Z (last)": last_numeric(models.get_altman_z_score()),
}
valuation_df = pd.DataFrame.from_dict(valuation, orient="index", columns=["value"])
display(valuation_df)

try:
    intrinsic = models.get_intrinsic_valuation()
    display(intrinsic.tail() if isinstance(intrinsic, pd.DataFrame) else intrinsic)
except Exception as exc:
    print("Intrinsic valuation skipped:", exc)

## Phase 5 — Risk & Portfolio Context

In [ ]:
perf = ft_ta.performance
risk = ft_ta.risk

risk_kpis = {
    "Sharpe": last_numeric(perf.get_sharpe_ratio()),
    "Sortino": last_numeric(perf.get_sortino_ratio()),
    "Jensen Alpha": last_numeric(perf.get_jensens_alpha()),
    "Beta": last_numeric(perf.get_beta()),
    "VaR": last_numeric(risk.get_value_at_risk()),
    "CVaR": last_numeric(risk.get_conditional_value_at_risk()),
    "Max Drawdown": last_numeric(risk.get_maximum_drawdown()),
    "Ulcer Index": last_numeric(risk.get_ulcer_index()),
}
risk_kpi_df = pd.DataFrame.from_dict(risk_kpis, orient="index", columns=["value"])
display(risk_kpi_df)

## Phase 6 — Market Segment ETF + Peer Relative Analysis

In [ ]:
SECTOR_ETF_MAP = {
    "Technology": ["XLK", "VGT"],
    "Financial Services": ["XLF", "VFH"],
    "Financial": ["XLF", "VFH"],
    "Healthcare": ["XLV", "VHT"],
    "Health Care": ["XLV", "VHT"],
    "Industrials": ["XLI", "VIS"],
    "Consumer Cyclical": ["XLY", "VCR"],
    "Consumer Defensive": ["XLP", "VDC"],
    "Energy": ["XLE", "VDE"],
    "Basic Materials": ["XLB", "VAW"],
    "Utilities": ["XLU", "VPU"],
    "Real Estate": ["XLRE", "VNQ"],
    "Communication Services": ["XLC", "VOX"],
}

peer_candidates = []
if not peers_df.empty:
    peer_cols = [c for c in peers_df.columns if c.lower() in ["symbol", "peer", "ticker"]]
    if peer_cols:
        peer_candidates = peers_df[peer_cols[0]].dropna().astype(str).tolist()

peer_candidates = [p for p in peer_candidates if p.upper() != SYMBOL.upper()]
peer_candidates = list(dict.fromkeys(peer_candidates))[:10]

sector_etfs = SECTOR_ETF_MAP.get(str(sector), ["SPY"])
universe = list(dict.fromkeys([SYMBOL] + peer_candidates + sector_etfs + [BENCHMARK]))

print("Sector:", sector)
print("Peer candidates:", peer_candidates)
print("ETF benchmark basket:", sector_etfs)
print("Universe:", universe)

In [ ]:
hist_obj, hist_provider = call_obb(
    obb.equity.price.historical,
    symbol=",".join(universe),
    start_date=START_DATE_TECHNICALS,
    end_date=END_DATE,
    interval="1d",
)
hist_df = to_df(hist_obj)
print("Historical data provider:", hist_provider, "rows:", len(hist_df))
display(hist_df.head())

### Relative Metrics Computation
Build aligned close series and compute return/risk metrics for the selected universe.

In [ ]:
if {"date", "symbol", "close"}.issubset(set(hist_df.columns)):
    close = hist_df.pivot(index="date", columns="symbol", values="close").sort_index()
else:
    close = hist_df.copy()

close = close.dropna(how="all")
returns = close.pct_change().dropna(how="all")

annual_ret = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
sharpe = (annual_ret - RISK_FREE_RATE) / annual_vol
var_95 = returns.quantile(0.05)
cvar_95 = returns.where(returns.le(var_95), np.nan).mean()

drawdown = (1 + returns).cumprod() / (1 + returns).cumprod().cummax() - 1
max_drawdown = drawdown.min()

relative_table = pd.DataFrame({
    "Annual Return": annual_ret,
    "Volatility": annual_vol,
    "Sharpe": sharpe,
    "VaR 95%": var_95,
    "CVaR 95%": cvar_95,
    "Max Drawdown": max_drawdown,
}).sort_values("Sharpe", ascending=False)

display(relative_table)

### Relative Analysis Visuals
Correlation heatmap and risk-return scatter for peer and benchmark comparison.

In [ ]:
corr = returns.corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Asset Returns")
plt.show()

rr = relative_table.reset_index().rename(columns={"index": "symbol"})
plt.figure(figsize=(9, 6))
sns.scatterplot(data=rr, x="Volatility", y="Annual Return", size="Sharpe", hue="Sharpe", sizes=(80, 400), palette="viridis")
for _, row in rr.iterrows():
    plt.text(row["Volatility"], row["Annual Return"], str(row["symbol"]))
plt.title("Risk vs Return (size/color = Sharpe)")
plt.show()

## Phase 7 — Decision, Execution, and Monitoring

### Decision Scoring Template
Populate phase scores and generate final Buy/Hold/Sell decision label.

In [ ]:
scores = {
    "business_quality": np.nan,
    "fundamentals": np.nan,
    "technicals": np.nan,
    "valuation": np.nan,
    "risk_fit": np.nan,
    "relative_peer_score": np.nan,
}
weights = {
    "business_quality": 0.08,
    "fundamentals": 0.25,
    "technicals": 0.15,
    "valuation": 0.20,
    "risk_fit": 0.12,
    "relative_peer_score": 0.20,
}

if all(pd.notna(v) for v in scores.values()):
    total_score = sum(scores[k] * weights[k] for k in scores)
else:
    total_score = np.nan

def decision_label(score):
    if pd.isna(score):
        return "Pending"
    if score >= 4.2:
        return "Strong Buy"
    if score >= 3.6:
        return "Buy"
    if score >= 2.8:
        return "Hold/Watch"
    return "Avoid/Sell"

decision_df = pd.DataFrame([
    {
        "symbol": SYMBOL,
        "total_score": total_score,
        "decision": decision_label(total_score),
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
])

display(pd.DataFrame(scores, index=["score"]).T.rename(columns={"score": "value"}))
display(decision_df)

## Analyst Notes
- Fill subjective score cells after reviewing outputs from all phases.
- Keep bull/bear thesis and invalidation conditions in your final decision memo.
- Re-run after earnings or major macro/sector regime changes.